## Setup

In [ ]:
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

## Invariants

In [ ]:
def right_cauchy_green(F):
    return F.T @ F

def isochoric_C(C):
    J = jnp.sqrt(jnp.linalg.det(C)) # linalg is short for linear algebra. jnp.linalg is the part of jax.numpy that contains linear algebra functions. jnp.linalg.det computes the determinant of a matrix.
    return J ** (-2.0/3.0) * C, J
    
def invariants(C_bar):
    I1_bar = jnp.trace(C_bar)
    I2_bar = jnp.trace(jnp.linalg.inv(C_bar)) # tr(C_bar^{-1}), not the polynomial invariant I2_bar = 0.5 * (I1_bar**2 - tr(C_bar^2))
    return I1_bar, I2_bar

The branch update will later need `psi_corrected` as well. At that point we'll move all shared functions into one `.py` file and import it, so nothing is copied twice. 

## Time constants

In [ ]:
N_MB = 5
T_K = jnp.logspace(-2, 2, N_MB) 
print(T_K)

In [ ]:
def init_tau_params(key, hidden=(8,8)): 
    keys = jax.random.split(key, len(hidden) + 2) # keys is a list of random keys, one for each layer of the neural network. +2 accounts for the input and output layers. jax.random.split splits a random key into multiple keys.
    params = {
        "W_defo": 0.1 * jax.random.normal(keys[0], (hidden[0], 2)), # W_defo is the weight matrix for the first layer of the neural network. It has shape (hidden[0], 2) because the input layer has 2 neurons (I1_bar and I2_bar) and the first hidden layer has hidden[0] neurons. jax.random.normal generates random numbers from a normal distribution.
        "w_T": 0.1 * jax.random.normal(keys[1], (hidden[0],)), # w_T is the weight matrix for the first layer of the neural network. It has shape (hidden[0],) because the input layer has 1 neuron (T) and the first hidden layer has hidden[0] neurons. jax.random.normal generates random numbers from a normal distribution.    
        "b0": jnp.zeros(hidden[0]), 
        "layers": [],
    }
    for i in range(1, len(hidden)):
        params["layers"].append({
            "W": 0.1 * jax.random.normal(keys[i+1], (hidden[i], hidden[i-1])), # W is the weight matrix for the i-th layer of the neural network. It has shape (hidden[i], hidden[i-1]) because the i-th hidden layer has hidden[i] neurons and the (i-1)-th hidden layer has hidden[i-1] neurons. jax.random.normal generates random numbers from a normal distribution.
            "b": jnp.zeros(hidden[i]), # b is the bias vector for the i-th layer of the neural network. It has shape (hidden[i],) because the i-th hidden layer has hidden[i] neurons.
        })
    params["w_out"] = 0.1 * jax.random.normal(keys[-1], (hidden[-1], )) # w_out is the weight matrix for the output layer of the neural network. It has shape (hidden[-1], 1) because the last hidden layer has hidden[-1] neurons and the output layer has 1 neuron (psi). jax.random.normal generates random numbers from a normal distribution.
    return params

key_tau = jax.random.PRNGKey(1)
branch_keys = jax.random.split(key_tau, N_MB)
tau_params = [init_tau_params(k) for k in branch_keys] # tau_params is a list of dictionaries, one for each branch of the neural network. Each dictionary contains the parameters for that branch.

print(len(tau_params))
print(tau_params[0]["W_defo"].shape, tau_params[0]["w_T"].shape)

leaves = jax.tree_util.tree_leaves(tau_params[0])
print(len(leaves), [l.shape for l in leaves])
print(sum([l.size for l in leaves]))

## Forward Pass

$$
\mathcal N_k = \mathbf w_{\mathrm{out}}\cdot\mathbf z_L, \qquad \tau_k = T_k\,\operatorname{softplus}(\mathcal N_k)
$$

Remember we need the relaxation times to be positive to avoid instabilities. Also it does not make physical sense for a time to be negative.

In [ ]:
positive = jax.nn.softplus 
act_fn = jax.nn.softplus 

def tau_raw(params, I1_bar, I2_bar, T_hat):
    defo = jnp.array([I1_bar - 3.0, I2_bar - 3.0]) 
    z = act_fn(params["W_defo"] @ defo + params["w_T"] * T_hat + params["b0"]) 
    for layer in params["layers"]:
        z = act_fn(layer["W"] @ z + layer["b"]) 
    return params["w_out"] @ z 

def tau_k(params, T_k, C_bar, T_hat):
    I1_bar, I2_bar = invariants(C_bar)
    return T_k * positive(tau_raw(params, I1_bar, I2_bar, T_hat))

I am going to use `zip(tau_params, T_K)` to walk through the two lists in parallel. It will give `(params of branch 1, T_1)`, then `(params of brach 2, T_2)`, and so on. The printed ration is the network's factor:

$$
\frac{\tau_k}{T_k} = \operatorname{softplus}(\mathcal N_k)
$$

In [ ]:
C_bar_ref = jnp.eye(3) 

for T_hat in (-1.0, 0.0, 1.0):
    taus = jnp.array([tau_k(p, Tk, C_bar_ref, T_hat) for p, Tk in zip(tau_params, T_K)])
    print("----------------------------------------------------")
    print(f"T_hat = {T_hat:+.1f}")
    print(" tau_k = ", taus)
    print(" Ratios tau_k / T_k = ", taus / T_K)
    print(" all positive:", bool(jnp.all(taus > 0.0)))


Let's check the inversion against the network itself and see how large the output weights really are.

$$
\mathcal N_k = \operatorname{softplus}^{-1}\!\left(\frac{\tau_k}{T_k}\right) = \ln\!\left(e^{\tau_k/T_k} - 1\right)
$$

In [ ]:
def softplus_inv(y):
    return jnp.log(jnp.expm1(y))      # ln(e^y - 1); expm1 is accurate for small y

C_bar_ref = jnp.eye(3)
for k, (p, Tk) in enumerate(zip(tau_params, T_K), start=1):
    N_direct   = tau_raw(p, 3.0, 3.0, 0.0)
    N_inverted = softplus_inv(tau_k(p, Tk, C_bar_ref, 0.0) / Tk)
    print(f"branch {k}: N_k = {N_direct:+.5f}   from inversion: {N_inverted:+.5f}   "
          f"max|w_out| = {jnp.abs(p['w_out']).max():.3f}")